# Recurring subscription definition and feature sensitivity analysis

**Research question.** How do alternative definitions of *recurrence* change estimated monthly subscription
spending, and do stricter definitions separate ongoing financial commitments from repeated discretionary
purchases? This is exploratory feature validation. It may inform later financial-distress modelling, but **no distress
prediction has been conducted in this notebook**.

**Two different things are called "baseline": keep them apart**

| name | what it is | module |
|---|---|---|
| **PA7 baseline classifier** | the original PA7 keyword rules that decide which debits are subscription-type (preserved verbatim) | `subrec/rules_baseline.py` |
| **M0** | the original PA7 *recurrence rule*: ≥ 3 distinct charged months anywhere in the account history. A recurrence definition, not a classifier; it uses future months | `subrec/recurrence.py` |
| **v2 classifier** | alternative keyword classifier, provisional pending human review | `subrec/rules_v2.py` |
| **M1, M2, M3** | new recurrence definitions that use only the present and past | `subrec/recurrence.py` |

PA7's original feature = PA7 baseline classifier + M0 (reproduced exactly by a test). Every other combination is new.

**Reading the results.** Stricter recurrence definitions (M2, M3) count fewer charges. That does **not** make them more
accurate: no definition has been checked against human labels yet (see section 7).

**What this notebook does**

1. Classifies debits with the preserved PA7 baseline classifier (`baseline` in tables) and an alternative classifier (**v2**) that
   fixes unreachable subtype keywords, removes obvious false positives, and separates high-confidence
   subscription merchants from ambiguous billing platforms (Apple, Google, Microsoft ...).
2. Applies four recurrence definitions (M0, M1, M2, M3 at 10 / 15 / 20 %) at the (account, merchant, month) level.
3. Builds account-month features with the existing coverage methodology (covered / no_data_gap / partial_edge).
4. Compares the definitions on a common eligible population, with sensitivity to ambiguous platforms,
   to exact-repeated records, and to the gap policy.
5. Draws a stratified human-review sample for future validation and writes `analysis_summary.md`.

**Primary specification (research-lead decisions, 2026-11-24)**

* Partial edge months are **not** recurrence history for M1, M2 or M3. M0 is unchanged (retrospective benchmark).
* A month with no data breaks consecutive-month windows (`gap_policy="break"`); `bridge` is a sensitivity check.
* M3 primary = **15 %** median-based stability; 10 % and 20 % are sensitivity checks.
* Merchant grouping stays platform-level (limitations flagged in 5b and section 10).
* Primary view: **v2 classifier (provisional, pending human review) · original records · common eligible
  population · ambiguous platforms excluded**. Ambiguous platforms *included* is the sensitivity analysis.

**How to run**

| | Mac (synthetic) | DSMLP (real) |
|---|---|---|
| mode | default, `SUBREC_MODE=synthetic` | `SUBREC_MODE=real` |
| data | generated by `subrec/synthetic.py` | `full_histories_5000accts.parquet` (written by `income_feature.ipynb`) |
| outputs | `outputs_subscription_recurrence/synthetic/` | `outputs_subscription_recurrence/real/` |

Start Jupyter from this folder (`EDA/subscription_recurrence/`). Set the environment variables before starting
Jupyter, or edit the configuration cell. Run top to bottom.
All reusable code lives in `subrec/`; tests live in `tests/`.

> **Privacy.** In real mode no memo text is printed into this notebook. Memo-level files (classifier
> disagreements, the human-review sample) go to `PRIVATE_DIR` and must stay on DSMLP.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path

MODE = os.environ.get("SUBREC_MODE", "synthetic")   # DSMLP: change to "real" here, or export SUBREC_MODE=real before starting Jupyter
assert MODE in ("synthetic", "real"), MODE
PROJECT_DIR = Path(os.environ.get("SUBREC_PROJECT_DIR", Path.cwd())).resolve()
HIST_CACHE_PATH = ""   # DSMLP real mode: full path of the existing full_histories_5000accts.parquet (or export SUBREC_HIST_CACHE)
HIST_CACHE = Path(os.environ.get("SUBREC_HIST_CACHE") or HIST_CACHE_PATH or PROJECT_DIR / "full_histories_5000accts.parquet")
OUT_DIR = Path(os.environ.get("SUBREC_OUT_DIR", PROJECT_DIR / "outputs_subscription_recurrence")) / MODE
PRIVATE_DIR = Path(os.environ.get("SUBREC_PRIVATE_DIR", OUT_DIR / "private"))   # memo-level files: keep on DSMLP
FIG_DIR = OUT_DIR / "figures"
for d in (OUT_DIR, PRIVATE_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 0
N_RANDOM_ACCOUNTS = 40            # synthetic mode only
REVIEW_PER_STRATUM = 15
SHOW_MEMOS = MODE == "synthetic"  # never print real memo text into notebook outputs
RUN_TESTS = MODE == "synthetic"

sys.path.insert(0, str(PROJECT_DIR))
from subrec.config import RecurrenceConfig, FAKE_ACCOUNT
CFG = RecurrenceConfig(min_months=3, window=3, m3_thresholds={"M3_10": .10, "M3_15": .15, "M3_20": .20},
                       gap_policy="break", history_requires_covered=True)
print(f"MODE={MODE}\nPROJECT_DIR={PROJECT_DIR}\nOUT_DIR={OUT_DIR}\n{CFG}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from subrec.data import load_history, prepare_history, exact_distinct, repeated_record_diagnostics
from subrec.recurrence import label_charges
from subrec.pipeline import run_analysis, run_one, status_dollars
from subrec.metrics import metrics_table, compare_variants
from subrec.review import build_review_sample, evaluate_review
from subrec import plots
from subrec.report import build_summary, write_summary

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
METHODS = CFG.methods
PRIMARY = dict(dataset="original", classifier="v2", platforms="excluded", population="common")   # primary specification
print("pandas", pd.__version__, "| numpy", np.__version__, "| matplotlib", matplotlib.__version__)

## 1. Data

Synthetic mode builds 15 scripted edge-case accounts (monthly, irregular, annual, price change, paused,
missing month, repeated records, gambling / events, ambiguous platform, whole-account copies, new subscription,
false-positive memos, missing keywords, fee memos, short account), the known synthetic account (`FAKE_ACCOUNT` in `subrec/config.py`), and
random accounts. Real mode reads the existing cache; it never scans the restricted share.

The known synthetic account is excluded in both modes. Records are **not** removed or collapsed in the primary analysis.

In [ ]:
if MODE == "synthetic":
    from subrec.synthetic import make_synthetic_history
    raw = make_synthetic_history(seed=SEED, n_random_accounts=N_RANDOM_ACCOUNTS)
    source = f"synthetic (seed={SEED}, random accounts={N_RANDOM_ACCOUNTS})"
else:
    raw = load_history(HIST_CACHE)
    source = f"cache {HIST_CACHE.name}"

hist = prepare_history(raw)
meta = {"source": source, "rows": len(hist), "accounts": int(hist.account_key.nunique()),
        "date_min": str(hist.date.min().date()), "date_max": str(hist.date.max().date()),
        "rows_before_fake_exclusion": len(raw), "fake_account_rows_removed": int((raw.account_key == FAKE_ACCOUNT).sum())}
pd.Series(meta).to_frame("value")

### 1a. Data quality: exact repeated records

A record *repeats an existing combination* when another row has the same account, posted date, amount,
direction and memo. There is no transaction id, so a repeat can be an export artefact **or** a genuine
second identical purchase (two coffees, a split payment). These counts describe the data. Repeated rows are **not**
treated as confirmed duplicates and are not assumed to be errors. The primary analysis keeps every row; section 5c re-runs everything on
exact-distinct records as a sensitivity check.

In [ ]:
dup_overall, dup_copies, dup_per_acct = repeated_record_diagnostics(hist)
display(pd.Series(dup_overall).to_frame("value"))
display(dup_copies.head(10).rename_axis("copies of one combination"))
print("per-account repeat share, quantiles:")
print(dup_per_acct.quantile([.1, .25, .5, .75, .9]).round(3).to_dict())
hist_distinct = exact_distinct(hist)
print(f"exact-distinct rows: {len(hist_distinct):,} of {len(hist):,}")

## 2. Subscription classification

* **PA7 baseline classifier** (`baseline`): pa7 rules, copied verbatim into `subrec/rules_baseline.py` (a test checks the copy against
  pa7 cell 3, and that baseline + M0 reproduces pa7's `recurring_sub_spend` exactly).
* **v2** (`subrec/rules_v2.py`): direct merchant dictionary, first match wins:
  bank-fee memos, gambling, advertising → excluded; named services → `high_confidence`
  (streaming, software / apps, gym); one-time events, P2P / wallet transfers, retail stores → excluded;
  billing platforms → `ambiguous_platform`.

A classifier match only means *subscription-type merchant*. Whether a charge is recurring is decided by the
recurrence methods in section 4, never by the classifier.

In [ ]:
lab = {c: label_charges(hist, c) for c in ["baseline", "v2"]}

def scope(df):
    return (df[df.subtype.notna()].assign(in_feature=lambda d: d.in_feature)
              .groupby(["subtype", "in_feature"], dropna=False)
              .agg(rows=("amount", "size"), dollars=("amount", "sum"), accounts=("account_key", "nunique"))
              .sort_values("dollars", ascending=False).round(2))

for c, d in lab.items():
    display(Markdown(f"**{c}: what each subtype holds (debits, original records)**"))
    display(scope(d))
t = lab["v2"][lab["v2"].in_feature]
display(Markdown("**v2 tiers**"))
display(t.groupby("tier").agg(rows=("amount", "size"), dollars=("amount", "sum"), merchants=("merchant", "nunique")).round(2))

In [ ]:
# Where the two classifiers disagree (by dollars). Memo text only shown in synthetic mode.
cmp = lab["baseline"][["text", "memo", "amount"]].assign(
    base=lab["baseline"].subtype.where(lab["baseline"].in_feature, "not in feature"),
    v2=lab["v2"].subtype.where(lab["v2"].in_feature, lab["v2"].subtype.fillna("unmatched")))
dis = cmp[cmp.base != cmp.v2]
dis_tab = (dis.groupby(["base", "v2"]).agg(rows=("amount", "size"), dollars=("amount", "sum"),
                                           distinct_memos=("text", "nunique"))
              .sort_values("dollars", ascending=False).round(2))
display(dis_tab.head(20))
dis_ex = (dis.groupby(["base", "v2", "memo"]).agg(rows=("amount", "size"), dollars=("amount", "sum"))
             .reset_index().sort_values("dollars", ascending=False))
dis_ex.to_csv(PRIVATE_DIR / "classifier_disagreements.csv", index=False)
if SHOW_MEMOS:
    display(dis_ex.head(15))
print("memo-level disagreements ->", PRIVATE_DIR / "classifier_disagreements.csv")

## 3. Account-month coverage (unchanged methodology)

| status | meaning | primary measurement |
|---|---|---|
| `covered` | full month with ≥ 1 transaction of any kind | yes: $0 is a real zero |
| `no_data_gap` | no transactions at all inside the account's span | no: missing coverage, not zero |
| `partial_edge` | first or last month of the account | no |

**Populations.** *covered* = all covered months (pa7's denominator). *common* = covered months whose
previous `window − 1` history months are also observable (covered), i.e. months where every method,
including M2 / M3, could in principle confirm a subscription. The common population is the same set of
account-months for every method, classifier and dataset, so it is the primary comparison population.

## 4. Recurrence definitions

Decisions are made per (account, merchant, month) = the sum of that account's charges from that merchant in
that month. Statuses:

* **confirmed**: recurring at month *t* under the method → counted in `rec_<M>`.
* **potential**: subscription-type charge, but the history observed up to *t* is too short to decide (merchant
  first seen inside the window, or window months unobserved / partial) → counted separately in `pot_<M>`.
* **not_confirmed**: enough history was observable and the rule failed (irregular, paused, amount unstable).

| method | rule at month *t* | uses future? |
|---|---|---|
| **M0** | ≥ 3 distinct charged months anywhere in the account history (pa7) | **yes**: retrospective comparison only |
| **M1** | ≥ 3 distinct charged **covered** months up to and including *t* (partial edge months are not history) | no |
| **M2** | a charge in each of *t − 2, t − 1, t*, with *t − 2* and *t − 1* covered | no |
| **M3_x** | M2 and every monthly total in the window within *x* of the window median. **M3_15 is primary**; M3_10 and M3_20 are sensitivity checks | no |

**Partial edge months are not recurrence history for M1, M2 or M3** (`history_requires_covered=True`); M0 is unchanged.

Gap policy (`CFG.gap_policy`): `break` (**primary**) uses consecutive *calendar* months, so a `no_data_gap` month
makes the window unobservable (charges after it are *potential* for two months). `bridge` skips gap months (sensitivity check).
`annual_candidate` (charge 12 months earlier, similar amount) is a diagnostic flag, not a method.

In [ ]:
A = run_analysis({"original": hist, "exact_distinct": hist_distinct}, CFG)
grid = A.runs[("original", "v2")].grid
coverage_counts = grid["status"].value_counts().rename("account_months").to_frame()
coverage_counts["share"] = (coverage_counts.account_months / len(grid)).round(3)
display(coverage_counts)
print("common population account-months:", int(grid.eligible_common.sum()),
      "| covered:", int(grid.eligible_covered.sum()))

# gap-policy sensitivity (v2, original records)
CFG_BRIDGE = RecurrenceConfig(**{**CFG.__dict__, "gap_policy": "bridge"})
r_bridge = run_one(hist, "original", "v2", CFG_BRIDGE)

## 5. Evaluation

### 5a. Recurrence definitions, primary view
v2 classifier (provisional), original records, **ambiguous platforms excluded**, common population: the primary specification. M3_15 is the primary M3 variant. Percentiles labelled
`_all` are over all eligible account-months (zeros included); `_positive` only over months with spend > 0.

In [ ]:
m = A.metrics
SHOW = ["eligible_account_months", "accounts_detected", "pct_accounts_detected", "zero_share", "median_all",
        "median_positive", "p90_all", "p95_all", "p99_all", "p90_positive", "p95_positive", "p99_positive",
        "total_recurring", "ambiguous_share", "potential_total"]

def view(**kw):
    x = m
    for k, v in kw.items():
        x = x[x[k] == v]
    return x.set_index("method").loc[METHODS]

primary = view(**PRIMARY)
display(primary[SHOW].round(3))
display(Markdown("**Same, all covered months (pa7 denominator)**"))
display(view(dataset="original", classifier="v2", platforms="excluded", population="covered")[SHOW].round(3))

### 5b. Ambiguous platform merchants (sensitivity) and classifier choice

Primary results exclude ambiguous platforms (Apple, Google, Microsoft, Xbox, PlayStation, Nintendo, Amazon digital,
Steam, Roku, generic YouTube / Disney); the table compares them with the *included* variant.

**Limitation of platform-level grouping.** Merchant identity is the canonical platform, so several products billed
through one platform (two Apple subscriptions, a subscription plus an app purchase) form one series whose monthly
total is a sum. That can fail M3 for genuine subscriptions or pass it for coincidental sums. The diagnostic below
shows how many confirmed platform months contain more than one charge.

In [ ]:
inc = view(dataset="original", classifier="v2", platforms="included", population="common")
amb_tab = pd.DataFrame({
    "% accounts excl. (primary)": primary["pct_accounts_detected"],
    "% accounts incl.": inc["pct_accounts_detected"],
    "zero share excl. (primary)": primary["zero_share"],
    "zero share incl.": inc["zero_share"],
    "total excl. (primary)": primary["total_recurring"],
    "total incl.": inc["total_recurring"],
    "ambiguous share of incl. total": inc["ambiguous_share"],
    "baseline ambiguous share (pa7 cues)": view(dataset="original", classifier="baseline", platforms="included", population="common")["ambiguous_share"],
})
display(amb_tab.round(3))

pm_prim = A.runs[("original", "v2")].pm
plat = pm_prim[pm_prim.is_ambiguous & pm_prim.status.eq("covered")]
rows = []
for k in METHODS:
    c = plat[plat[f"status_{k}"] == "confirmed"]
    rows.append({"method": k, "confirmed platform pair-months": len(c),
                 "share with more than one charge in the month": (c.n_charges > 1).mean() if len(c) else float("nan")})
display(Markdown("**Platform-grouping diagnostic (covered months)**"))
display(pd.DataFrame(rows).set_index("method").round(3))
display(Markdown("**Baseline (pa7) vs v2 classifier, ambiguous platforms included as in pa7**"))
display(compare_variants(m[(m.dataset == "original") & (m.platforms == "included") & (m.population == "common")],
                         by="classifier", left="baseline", right="v2").round(3))

### 5c. Original vs exact-distinct records
Same pipeline on exact-distinct records. Coverage statuses are identical by construction (removing exact
repeats never empties a month), so the common population is the same set of account-months. Differences are
a sensitivity range, not a correction.

In [ ]:
dd = compare_variants(m[(m.classifier == "v2") & (m.platforms == "excluded") & (m.population == "common")],
                      by="dataset", left="original", right="exact_distinct")
display(dd.round(3))
assert (A.runs[("original", "v2")].grid["status"].values == A.runs[("exact_distinct", "v2")].grid["status"].values).all()

### 5d. Gap policy and status composition

In [ ]:
gap_table = pd.concat({
    "break": primary[SHOW],
    "bridge": metrics_table(r_bridge.am_high, METHODS, ("common",)).set_index("method").loc[METHODS, SHOW],
}).round(3)
display(gap_table)
pm_v2 = A.runs[("original", "v2")].pm
status_table = status_dollars(pm_v2[~pm_v2.is_ambiguous], METHODS)
display(Markdown("**Subscription-merchant dollars in covered months by status (v2, original, ambiguous platforms excluded)**"))
display(status_table.round(2))
pm_v2 = A.runs[("original", "v2")].pm
print("annual_candidate pair-months (diagnostic):", int(pm_v2.annual_candidate.sum()),
      "| of which not confirmed by M2:", int((pm_v2.annual_candidate & pm_v2.status_M2.ne("confirmed")).sum()))

## 6. Figures

In [ ]:
sub = f"{MODE} data · v2 classifier · original records · platforms excluded · common population"
figs = {
    "01_method_overview.png": plots.method_overview(m, METHODS, sub, dataset="original", classifier="v2",
                                                    platforms="excluded", population="common"),
    "02_spending_distribution.png": plots.spending_distribution(A.runs[("original", "v2")].am_high, METHODS,
                                                                 "eligible_common", sub),
    "03_platform_sensitivity.png": plots.platform_sensitivity(m, METHODS, f"{MODE} data · original records · common population",
                                                              dataset="original", population="common"),
    "04_repeated_records_sensitivity.png": plots.two_way(m, METHODS, "dataset", ["original", "exact_distinct"],
        [("total_recurring", "Total confirmed recurring spend", plots.MONEY),
         ("pct_accounts_detected", "Accounts with a detected subscription", plots.PCT),
         ("median_positive", "Median spend, positive months", plots.MONEY)],
        f"{MODE} data · v2 · platforms excluded · common population",
        classifier="v2", platforms="excluded", population="common"),
    "05_classifier_comparison.png": plots.two_way(m, METHODS, "classifier", ["baseline", "v2"],
        [("total_recurring", "Total confirmed recurring spend", plots.MONEY),
         ("pct_accounts_detected", "Accounts with a detected subscription", plots.PCT)],
        f"{MODE} data · original records · platforms included · common population",
        dataset="original", platforms="included", population="common"),
    "06_status_composition.png": plots.status_composition(status_table, METHODS[1:],
                                                          f"{MODE} data · v2 · original records · platforms excluded"),
}
for name, fig in figs.items():
    fig.savefig(FIG_DIR / name, dpi=150, facecolor=fig.get_facecolor())
plt.show()

## 7. Human-review sampling (for future validation)

Nothing here shows that a stricter definition is more accurate: stricter rules change *what is counted*.
Validation needs human labels.

**Procedure**
1. `build_review_sample` draws up to `REVIEW_PER_STRATUM` covered pair-months per stratum, where stratum =
   merchant tier × the pattern of decisions by M0 / M1 / M2 / M3_15 (e.g. `M0+ M1+ M2- M3_15-`).
   Disagreements are over-sampled on purpose; each row carries `sample_weight` = stratum size / rows drawn.
2. Two reviewers independently fill `human_label` (`recurring` = an ongoing plan or membership, `not_recurring`,
   `unsure`) and `human_cadence` from the memo examples and the amount history (the current month is in
   brackets). Reviewers do not look at the method columns.
3. Resolve disagreements, report Cohen's kappa (`cohen_kappa`), then score with `evaluate_review`: weighted
   precision / recall per method. Recall is relative to classifier-flagged charges only.
4. The sample contains memo text: it is written to `PRIVATE_DIR` and stays on DSMLP.

In [ ]:
r = A.runs[("original", "v2")]
review = build_review_sample(r.pm, r.charges, n_per_stratum=REVIEW_PER_STRATUM, seed=SEED)
review_path = PRIVATE_DIR / "review_sample_v2.csv"
review.to_csv(review_path, index=False)
print(f"{len(review)} rows in {review.stratum.nunique()} strata -> {review_path}")
display(review.groupby("stratum").agg(sampled=("review_id", "size"), stratum_size=("stratum_size", "first")))

if MODE == "synthetic":
    # Pipeline check only: label the sample from the generator's ground truth and score it.
    # Circular by design (we wrote the truth); it shows the scoring code works, not which method is better.
    truth = (r.charges[r.charges.in_feature]
             .assign(sub=lambda d: d.truth.isin(["subscription_monthly", "platform_subscription", "subscription_annual"]))
             .groupby(["account_key", "merchant", "month"])["sub"].any())
    demo = review.copy()
    demo["human_label"] = [("recurring" if truth.get((a, mer, pd.Period(mo, "M")), False) else "not_recurring")
                           for a, mer, mo in zip(demo.account_key, demo.merchant, demo.month)]
    display(Markdown("**Synthetic oracle scoring (pipeline check, not evidence)**"))
    display(evaluate_review(demo).round(3))

## 8. Synthetic tests

Runs `pytest tests/` (synthetic mode only) and saves the output next to the other results. Tests cover monthly
subscriptions, irregular purchases, annual plans, price changes, paused subscriptions, missing months, repeated
records, future-data leakage (truncation and injected future charges), classifier rules, exact reproduction of the
pa7 baseline, review sampling and the summary generator.

In [ ]:
if RUN_TESTS:
    res = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no", str(PROJECT_DIR / "tests")],
                         cwd=PROJECT_DIR, capture_output=True, text=True)
    (OUT_DIR / "test_results.txt").write_text(res.stdout + res.stderr)
    print(res.stdout[-1500:])
    assert res.returncode == 0, "synthetic tests failed"
else:
    print("tests skipped in real mode (run them on the Mac / in synthetic mode)")

## 9. Save outputs and write `analysis_summary.md`

`analysis_summary.md` contains computed tables only; the *Findings* section is left for the team. In synthetic mode
it carries a banner saying the numbers are not research results.

In [ ]:
m.to_csv(OUT_DIR / "metrics_all_variants.csv", index=False)
for tag, frame in [("primary_platforms_excluded", r.am_high), ("platforms_included", r.am)]:
    (frame.assign(month=frame.month.astype(str))
          .drop(columns=[c for c in frame.columns if c.startswith("prev")])
          .to_csv(OUT_DIR / f"account_month_recurring_v2_original_{tag}.csv", index=False))
pd.Series(dup_overall).to_json(OUT_DIR / "repeated_record_diagnostics.json", indent=2)
gap_table.to_csv(OUT_DIR / "gap_policy_sensitivity.csv")
status_table.to_csv(OUT_DIR / "status_dollars_v2_original.csv")

summary = build_summary(A, MODE, meta, dup_overall, coverage_counts, gap_table=gap_table,
                        status_table=status_table, figures=[f"figures/{k}" for k in figs])
path = write_summary(OUT_DIR / "analysis_summary.md", summary)
print("wrote", path)
print(sorted(p.relative_to(OUT_DIR).as_posix() for p in OUT_DIR.rglob("*") if p.is_file()))

## 10. Decisions applied and limitations

**Decisions applied (research lead, 2026-11-24)**
* Partial edge months are not recurrence history for M1, M2 or M3; M0 is the unchanged retrospective benchmark.
* Missing-data months break consecutive-month windows (primary); `bridge` is a sensitivity check.
* M3 stability = max |monthly total − window median| / median; **15 %** primary, 10 % / 20 % sensitivity.
* Platform-level merchant grouping retained (limitations below).
* Primary view: v2 classifier, original records, common population, ambiguous platforms **excluded**; included = sensitivity.
* v2 classifier is **provisional**, pending human review.

**Limitations: read before interpreting any output**
* **No validation.** Stricter recurrence definitions are not shown to be more accurate; they change what is counted.
  Only the human-review procedure (section 7) can support an accuracy claim.
* **Repeated records.** Rows that repeat an existing (account, date, amount, direction, memo) combination are kept in
  the primary analysis. Without a transaction id we cannot tell export artefacts from genuine identical purchases, so
  they are *not* confirmed duplicates. Exact-distinct results are a sensitivity range, not a correction; repeats raise
  monthly totals and can make a stable subscription look like it changed price (failing M3).
* **Platform-level merchant grouping.** Several products billed through one platform share one series; the monthly total
  is a sum, so M3 can reject genuine subscriptions or accept coincidental sums (see the 5b diagnostic).
* **Retrospective coverage.** The covered / no_data_gap / partial_edge statuses use each account's full observed span; the
  account's last observed month is only known to be "last" in hindsight, and M0 uses future months. Neither would be
  available in real time.
* **No refund netting.** Only debits are summed; credits and refunds from subscription merchants are not subtracted
  (same as PA7), so spend can be overstated for accounts with refunds.
* Rules are unvalidated keyword patterns (both classifiers).
* Annual / quarterly plans are invisible to M1-M3 (diagnostic flag only); M0 mislabels annual plans as recurring.
* Charges inside a confirmed pair-month are all counted, even if one is a one-off from the same merchant.
* Accounts were sampled by activity; pooled account-month percentiles over-weight long, active accounts.